# Prepare Master Data

This notebook builds the project-level `master.csv` file used in the housing affordability analysis. It harmonizes the cleaned yearly HBS files, preserves the original COICOP housing category, constructs adjusted housing cost, and saves the adjusted master file for downstream analysis.


## Data Sources

The household survey files used here are the cleaned Iranian Household Budget Survey files for 1394-1403 published by D-Learn: <https://d-learn.ir/iran-hbs/>. The page describes the files as cleaned and harmonized CSV/XLSX versions of the Statistical Center of Iran Household Expenditure and Income Survey.

CPI information is taken from the World Bank indicator `FP.CPI.TOTL.ZG` for Iran: <https://data.worldbank.org/indicator/FP.CPI.TOTL.ZG?end=2024&locations=IR&start=1960>. The local workbook `CPI.xlsx` is used as the project copy for the CPI merge.

The repository should treat the files in this folder as derived working data. If raw or cleaned survey files are not redistributed, the links above identify where the source data can be obtained.


## Reproducibility

The notebooks assume that the working directory is `Presentation/Cleaned` and that the yearly files (`94.csv` to `99.csv` and `400.csv` to `403.csv`) are available in that folder. The preparation notebook creates `master.csv`; the analysis notebook then reads `master.csv` and `CPI.xlsx`.

Recommended execution order:
1. `00_prepare_master.ipynb`
2. `01_housing_affordability_analysis.ipynb`

`Analysis.ipynb` keeps the combined workflow in one file for convenience.


In [ ]:
# Shared definitions used across the analysis.
# Source-language strings appear only where they are needed to match raw data values.

from pathlib import Path

import numpy as np
import pandas as pd

DATA_DIR_CANDIDATES = [
    Path("../data/cleaned"),
    Path("data/cleaned"),
]
DATA_DIR = next(
    (candidate.resolve() for candidate in DATA_DIR_CANDIDATES if candidate.exists()),
    DATA_DIR_CANDIDATES[0].resolve(),
)
HOUSING_COL = "Housing, Water, Electricity, Gas and Other Fuels"
TEHRAN_LABELS = {"tehran", "تهران"}
OWNER_PATTERN = "own"
RENTER_PATTERN = "rent|mortg"


def clean_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Return a copy with stripped string column names."""
    out = df.copy()
    out.columns = out.columns.astype(str).str.strip()
    return out


def coerce_numeric(df: pd.DataFrame, columns: list[str], fill_missing: float | None = None) -> pd.DataFrame:
    """Coerce selected columns to numeric, optionally creating missing columns."""
    out = df.copy()
    for col in columns:
        if col not in out.columns:
            if fill_missing is None:
                raise KeyError(f"Missing required column: {col}")
            out[col] = fill_missing
        out[col] = pd.to_numeric(out[col], errors="coerce")
    return out


def weighted_mean(values: pd.Series, weights: pd.Series) -> float:
    """Weighted mean that ignores missing values and non-positive weights."""
    mask = values.notna() & weights.notna() & (weights > 0)
    if not mask.any():
        return np.nan
    return np.average(values[mask], weights=weights[mask])


def add_urban_rural_label(df: pd.DataFrame, source_col: str = "Urban_Rural", target_col: str = "UR") -> pd.DataFrame:
    out = df.copy()
    ur = out[source_col].astype(str).str.strip().str.lower()
    out[target_col] = pd.NA
    out.loc[ur.str.startswith("u"), target_col] = "Urban"
    out.loc[ur.str.startswith("r"), target_col] = "Rural"
    return out[out[target_col].isin(["Urban", "Rural"])].copy()


def add_tenure_group(df: pd.DataFrame, source_col: str = "Tenure", target_col: str = "tenure_group") -> pd.DataFrame:
    out = df.copy()
    tenure = out[source_col].astype(str).str.strip().str.lower()
    out[target_col] = pd.NA
    out.loc[tenure.str.contains(RENTER_PATTERN, na=False), target_col] = "Renter"
    out.loc[tenure.str.contains(OWNER_PATTERN, na=False), target_col] = "Owner"
    return out[out[target_col].isin(["Owner", "Renter"])].copy()


def add_urban_tehran_group(df: pd.DataFrame, ur_col: str = "UR", target_col: str = "UR_EXT") -> pd.DataFrame:
    out = df.copy()
    province = out["Province"].astype(str).str.strip().str.lower()
    out[target_col] = out[ur_col]
    out.loc[(out[ur_col] == "Urban") & province.isin(TEHRAN_LABELS), target_col] = "Urban_Tehran"
    return out


def infer_iran_year(file_name: str) -> int:
    stem = int(Path(file_name).stem)
    if stem >= 100:
        return 1000 + stem
    if stem >= 90:
        return 1300 + stem
    return 1400 + stem


## 1. Build the Master Analysis File

The yearly cleaned files are read from the working folder, harmonized to a common schema, and combined into `master`. The extract keeps only the identifiers, weights, geographic variables, tenure variables, expenditure categories, income variables, and decile measures required for the analysis.

The original COICOP housing column is retained at this stage. The adjusted housing-cost variable is constructed separately in the next step.


In [ ]:
# Build the reduced household-level analysis file.

FILES = [
    "94.csv", "95.csv", "96.csv", "97.csv", "98.csv", "99.csv",
    "00.csv", "01.csv", "02.csv", "03.csv",
    "400.csv", "401.csv", "402.csv", "403.csv",
]
missing_files = [file for file in FILES if not (DATA_DIR / file).exists()]
if missing_files:
    missing = ", ".join(missing_files)
    raise FileNotFoundError(
        f"Missing HBS input files in {DATA_DIR}: {missing}. "
        "Place the yearly CSV files in data/cleaned."
    )

BASE_KEEP_COLS = [
    "Year", "ID", "Urban_Rural", "Province", "County", "Weight",
    "Tenure", "Structure_Type", "House_Area",
    "Net_Expenditure", "Gross_Expenditure",
    "Food and Non-Alcoholic Beverages", "Clothing and Footwear", HOUSING_COL,
    "Furnishings Household Equipment and Routine Household Maintenance",
    "Health", "Transport", "Information and Communication",
    "Recreation and Culture", "Education Services",
    "Restaurants and Accommodation Services", "Other Commodities", "Transfer Payments",
    "Income", "Rent",
    "Income_Decile_OECD", "Income_Decile_OECD_Modified",
]

EXTRA_COLS = [
    "Cash_Rent",
    "NonCash_ImputedRent",
    "NonCash_ImputedRent_Mortgage",
    "NonCash_ImputedRent_Ownership",
]

ALIASES = {
    "Net_Expenditure": ["Net_Expenditure", "Net Expenditure"],
    "Gross_Expenditure": ["Gross_Expenditure", "Gross Expenditure", "Gross_Expenditure "],
    HOUSING_COL: [
        HOUSING_COL,
        "Housing, Water, Electricity, Gas and Other Fuels ",
        'Housing, Water, Electricity, Gas and Other Fuels"',
        '"Housing, Water, Electricity, Gas and Other Fuels"',
        "housing_cost_adj",
    ],
    "Cash_Rent": ["Cash_Rent", "Cash Rent"],
    "NonCash_ImputedRent": ["NonCash_ImputedRent", "NonCash ImputedRent"],
    "NonCash_ImputedRent_Mortgage": ["NonCash_ImputedRent_Mortgage", "NonCash ImputedRent Mortgage"],
    "NonCash_ImputedRent_Ownership": ["NonCash_ImputedRent_Ownership", "NonCash ImputedRent Ownership"],
}

KEEP_COLS = BASE_KEEP_COLS + EXTRA_COLS
frames = []
missing_report = []

for file_name in FILES:
    yearly = clean_columns(pd.read_csv(DATA_DIR / file_name))

    if "Year" not in yearly.columns:
        yearly["Year"] = infer_iran_year(file_name)

    for canonical, candidates in ALIASES.items():
        if canonical not in yearly.columns:
            match = next((col for col in candidates if col in yearly.columns), None)
            if match is not None:
                yearly = yearly.rename(columns={match: canonical})

    available_cols = [col for col in KEEP_COLS if col in yearly.columns]
    frame = yearly[available_cols].copy()
    frame["__source_file__"] = file_name
    frames.append(frame)

    missing_cols = [col for col in KEEP_COLS if col not in yearly.columns]
    if missing_cols:
        missing_report.append({"file": file_name, "missing_columns": missing_cols})

if not frames:
    raise FileNotFoundError("No yearly CSV files were found next to the notebook.")

master = pd.concat(frames, ignore_index=True)
master.to_csv(DATA_DIR / "master.csv", index=False)

print(f"master rebuilt: {master.shape[0]:,} rows x {master.shape[1]:,} columns")
print(f"years: {master['Year'].min()}-{master['Year'].max()}")
print(f"files used: {', '.join(FILES)}")

if missing_report:
    print("Some requested columns were not found in at least one file. Inspect missing_report for details.")


## 2. Adjust Housing Cost

This step reconstructs `housing_cost_adj` from the COICOP housing category. Imputed rent is removed for owner and mortgage households so that the resulting measure is closer to the cash and non-imputed housing burden used in the later comparisons.


In [ ]:
# Reconstruct adjusted housing cost.

df = clean_columns(master)

imputed_cols = [
    "NonCash_ImputedRent_Ownership",
    "NonCash_ImputedRent_Mortgage",
]
df = coerce_numeric(df, [HOUSING_COL] + imputed_cols, fill_missing=0.0)
df[imputed_cols] = df[imputed_cols].fillna(0)

tenure = df["Tenure"].astype(str).str.strip().str.lower()
is_owner = tenure.str.contains(OWNER_PATTERN, na=False)
is_mortgage = tenure.str.contains("mortg", na=False)

df["housing_cost_adj"] = df[HOUSING_COL].fillna(0)
df.loc[is_owner, "housing_cost_adj"] -= df.loc[is_owner, "NonCash_ImputedRent_Ownership"]
df.loc[is_mortgage, "housing_cost_adj"] -= df.loc[is_mortgage, "NonCash_ImputedRent_Mortgage"]
df["housing_cost_adj"] = df["housing_cost_adj"].clip(lower=0)

master["housing_cost_adj"] = df["housing_cost_adj"]

master.to_csv(DATA_DIR / "master.csv", index=False)

master[[
    HOUSING_COL,
    "NonCash_ImputedRent_Ownership",
    "NonCash_ImputedRent_Mortgage",
    "housing_cost_adj",
]].head()
